# Irradianza di una sorgente puntiforme
Fit 1: $V = A/(r-B)^2 + C$  |  Fit 2: $V = A/(r-B)^D + C$

## Librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

!pip install iminuit
from iminuit import Minuit

## Dati
Cambiare solo i valori con scritto CAMBIARE QUI.

In [ ]:
nome_file = "1_Irradianza_Foglio1___1_.csv"    # CAMBIARE QUI: nome del file csv

df = pd.read_csv(nome_file, sep=";", decimal=",")

# colonne del foglio (si conta da 0): 16 = x_old, 18 = V netta, 22 = sigma x, 23 = sigma V
dati = df.iloc[:, [16, 18, 22, 23]].dropna()
x_old = dati.iloc[:, 0].to_numpy(dtype=float)
y = dati.iloc[:, 1].to_numpy(dtype=float)
ex = dati.iloc[:, 2].to_numpy(dtype=float)
ey = dati.iloc[:, 3].to_numpy(dtype=float)
print("numero di punti:", len(y))

# distanza vera: r = (x - x0) + D/2 + 1 mm
x0 = 105.0       # CAMBIARE QUI: posizione a contatto (mm)
D_lamp = 16.683  # CAMBIARE QUI: diametro lampadina (mm)
x = (x_old - x0) + D_lamp/2 + 1

# errori sistematici comuni (servono solo per confrontare B con 0 alla fine)
sigma_x0 = 0.144    # mm
sigma_D = 0.05      # mm
sigma_1mm = 0.5     # mm (stima, da controllare)
sigma_sist = np.sqrt(sigma_x0**2 + (sigma_D/2)**2 + sigma_1mm**2)

## Fit 1
Gli errori su x vengono portati su y con la derivata (varianza efficace), come nello script del prof.

In [ ]:
def fitfunc1(x, A, B, C):
    return A / (x - B)**2 + C

def fitfuncprime(f, x, *par):
    dx = 1e-6
    return (f(x + dx/2, *par) - f(x - dx/2, *par)) / dx

def chi2_1(A, B, C):
    sigma = np.sqrt(ey**2 + (ex * fitfuncprime(fitfunc1, x, A, B, C))**2)
    return np.sum(((y - fitfunc1(x, A, B, C)) / sigma)**2)

# parametri iniziali: A dal primo punto, B e C a zero
m1 = Minuit(chi2_1, A=y[0]*x[0]**2, B=0.0, C=0.0)
m1.errordef = 1
m1.migrad()
m1.hesse()

A1, B1, C1 = m1.values["A"], m1.values["B"], m1.values["C"]
eA1, eB1, eC1 = m1.errors["A"], m1.errors["B"], m1.errors["C"]
chi2_min1 = m1.fval
ndof1 = len(x) - 3

print(f"A = {A1:5.5g} ± {eA1:5.2g}")
print(f"B = {B1:5.4g} ± {eB1:5.2g}")
print(f"C = {C1:5.4g} ± {eC1:5.2g}")
print(f"chi2/ndof = {chi2_min1:5.4g}/{ndof1}")

# B compatibile con 0? (errore statistico + sistematico)
errB_tot = np.sqrt(eB1**2 + sigma_sist**2)
print("B / errore =", B1 / errB_tot, "  (compatibile con 0 se < 2)")
# C compatibile con 0?
print("C / errore =", C1 / eC1, "  (compatibile con 0 se < 2)")

## Fit 2
Come il Fit 1 ma con l'esponente D libero. Si parte dai risultati del Fit 1.

In [ ]:
def fitfunc2(x, A, B, C, D):
    return A / np.abs(x - B)**D + C

def chi2_2(A, B, C, D):
    sigma = np.sqrt(ey**2 + (ex * fitfuncprime(fitfunc2, x, A, B, C, D))**2)
    return np.sum(((y - fitfunc2(x, A, B, C, D)) / sigma)**2)

m2 = Minuit(chi2_2, A=A1, B=B1, C=C1, D=2.0)
m2.errordef = 1
m2.migrad()
m2.hesse()

A2, B2, C2, D2 = m2.values["A"], m2.values["B"], m2.values["C"], m2.values["D"]
eA2, eB2, eC2, eD2 = m2.errors["A"], m2.errors["B"], m2.errors["C"], m2.errors["D"]
chi2_min2 = m2.fval
ndof2 = len(x) - 4

print(f"A = {A2:5.5g} ± {eA2:5.2g}")
print(f"B = {B2:5.4g} ± {eB2:5.2g}")
print(f"C = {C2:5.4g} ± {eC2:5.2g}")
print(f"D = {D2:5.4g} ± {eD2:5.2g}")
print(f"chi2/ndof = {chi2_min2:5.4g}/{ndof2}")

# D compatibile con 2?
print("(D - 2) / errore =", (D2 - 2) / eD2, "  (compatibile se < 2 in modulo)")

## Grafico 1: dati, fit e asintoti

In [ ]:
x_fit = np.linspace(min(x), max(x), 500)
y_fit = fitfunc1(x_fit, A1, B1, C1)

plt.figure()
plt.errorbar(x, y, xerr=ex, yerr=ey, fmt=".", color="k", label="dati")
plt.plot(x_fit, y_fit, "r-", label=f"$\\chi^2$/ndof = {chi2_min1:5.4g}/{ndof1}\nA = {A1:5.5g} $\\pm$ {eA1:5.2g}\nB = {B1:5.4g} $\\pm$ {eB1:5.2g}\nC = {C1:5.4g} $\\pm$ {eC1:5.2g}")
plt.axhline(C1, linestyle="--", color="b", label="asintoto orizzontale (C)")
plt.axvline(B1, linestyle="--", color="g", label="asintoto verticale (B)")

plt.title("Tensione in funzione della distanza")
plt.xlabel("r (mm)")
plt.ylabel("V (V)")
plt.legend()
plt.savefig("c1.png")
plt.show()

## Grafico 2: residui normalizzati
Se gli errori sono giusti dovrebbero seguire una gaussiana con media 0 e sigma 1.

In [ ]:
y_res = y - fitfunc1(x, A1, B1, C1)
ey_eff = np.sqrt(ey**2 + (ex * fitfuncprime(fitfunc1, x, A1, B1, C1))**2)
res_norm = y_res / ey_eff

plt.figure()
counts, bins, _ = plt.hist(res_norm, range=(-10, 10), bins=20, alpha=0.6)
centri = (bins[:-1] + bins[1:]) / 2

def gauss(x, c, media, sigma):
    return c * np.exp(-0.5 * ((x - media) / sigma)**2)

# fit gaussiano solo sui bin non vuoti
mask = counts > 0
popt, pcov = curve_fit(gauss, centri[mask], counts[mask], p0=[counts.max(), 0.0, 1.0],
                       sigma=np.sqrt(counts[mask]), absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))
chi2_h = np.sum(((counts[mask] - gauss(centri[mask], *popt)) / np.sqrt(counts[mask]))**2)
ndof_h = np.count_nonzero(counts) - 3

xg = np.linspace(-10, 10, 400)
plt.plot(xg, gauss(xg, *popt), "r-", label=f"$\\chi^2$/ndof = {chi2_h:5.4g}/{ndof_h}\nCostante = {popt[0]:5.4g} $\\pm$ {perr[0]:5.2g}\nMedia = {popt[1]:5.4g} $\\pm$ {perr[1]:5.2g}\nSigma = {abs(popt[2]):5.4g} $\\pm$ {perr[2]:5.2g}")

plt.title("Residui")
plt.xlabel("Residui normalizzati")
plt.ylabel("Conteggi")
plt.legend()
plt.savefig("c2.png")
plt.show()

print("media / errore =", popt[1] / perr[1], "  (atteso compatibile con 0)")
print("(sigma - 1) / errore =", (abs(popt[2]) - 1) / perr[2], "  (atteso compatibile con 1)")

## Grafico 3: misura contro atteso
I punti devono stare sulla retta misura = atteso.

In [ ]:
atteso = fitfunc1(x, A1, B1, C1)

plt.figure()
plt.errorbar(atteso, y, yerr=ey_eff, fmt=".", color="k", label="misure")
plt.plot([min(atteso), max(atteso)], [min(atteso), max(atteso)], "r-", label="misura = atteso")

# se i punti si ammassano vicino a 0 togliere il # alle due righe sotto
# plt.xscale("log")
# plt.yscale("log")

plt.title("Compatibilità misure")
plt.xlabel("Atteso (V)")
plt.ylabel("Misura (V)")
plt.legend()
plt.savefig("c3.png")
plt.show()